# Preparing the data 

In [1]:
import pandas as pd

In [2]:
gene_expression_path = '/mnt/bulk-saturn/maralampert/genhist/dataframes/tcga_RNASeq.csv'
pam50_path = '/mnt/bulk-saturn/maralampert/genhist/dataframes/brca_subtypes.csv'

In [3]:
# import csv file
# for the barcode file we want the column available (not as the index),
# otherwise subsequent operations like uppercasing or merging will fail
gene_expression_df = pd.read_csv(gene_expression_path, index_col=0)
pam50_df = pd.read_csv(pam50_path)

# inspect to make sure the barcode column is present
print('pam50 columns:', pam50_df.columns.tolist())

pam50 columns: ['bcr_patient_barcode', 'type', 'age_at_initial_pathologic_diagnosis', 'gender', 'race', 'ajcc_pathologic_tumor_stage', 'clinical_stage', 'histological_type', 'histological_grade', 'initial_pathologic_dx_year', 'menopause_status', 'birth_days_to', 'vital_status', 'tumor_status', 'last_contact_days_to', 'death_days_to', 'cause_of_death', 'new_tumor_event_type', 'new_tumor_event_site', 'new_tumor_event_site_other', 'new_tumor_event_dx_days_to', 'treatment_outcome_first_course', 'margin_status', 'residual_tumor', 'OS', 'OS.time', 'DSS', 'DSS.time', 'DFI', 'DFI.time', 'PFI', 'PFI.time', 'Redaction', 'NCI-T Label', 'NCI-T Code', 'Tumor_Sample_Barcode', 'all_counts', 'non_syn_counts', 'non_syn_tmb', 'bcr_sample_barcode', 'Majority_Subtype_mRNA', 'Exome_Covered', 'Exome_Unknown']


In [4]:
print(pam50_df.index.name)

None


In [5]:
# uppercase patient identifiers so merges are case-insensitive
gene_expression_df['Patient_ID'] = gene_expression_df['Patient_ID'].str.upper()

# make sure the barcode column exists and uppercase it too
pam50_df['bcr_patient_barcode'] = pam50_df['bcr_patient_barcode'].str.upper()

# now merge, including the subtype column if available
brca_df = pd.merge(
    gene_expression_df,
    pam50_df[['bcr_patient_barcode', 'Majority_Subtype_mRNA']],
    left_on='Patient_ID', right_on='bcr_patient_barcode',
    how='inner'
)

# drop helper columns; ignore errors if they are not present
brca_df = brca_df.drop(columns=['bcr_patient_barcode',
                                'Patient_ID_lower',
                                'Patient_ID_upper'],
                       errors='ignore')

In [ ]:
brca_df.head()

In [8]:
print(f'Number of rows in brca_df: {brca_df.shape[0]}')

Number of rows in brca_df: 996


In [9]:
print(f'Number of columns in brca_df: {brca_df.shape[1]}')

Number of columns in brca_df: 19964


In [10]:
# show value counts of 'Majority_Subtype_mRNA' column
print(brca_df['Majority_Subtype_mRNA'].value_counts())

Majority_Subtype_mRNA
LumA      514
LumB      195
Basal     175
Her2       74
Normal     38
Name: count, dtype: int64


In [11]:
# save brca_df to csv
brca_df.to_csv('../../data/brca_gene_expression_with_subtypes.csv', index=False)

In [12]:
#drop the 'Majority_Subtype_mRNA' column for the next steps
brca_df = brca_df.drop(columns=['Majority_Subtype_mRNA'])

In [14]:
# save brca_df to csv
brca_df.to_csv('../../data/brca_gene_expression.csv', index=False)

In [16]:
# check brca df for NaN values and say how many NaN values there are in total 
print(f'Total number of NaN values in brca_df: {brca_df.isna().sum().sum()}')

Total number of NaN values in brca_df: 0
